# Step 1 — Evaluate the base student

This notebook evaluates the pretrained checkpoint `Qwen/Qwen3-4B-Base` on the three MOPD domains: math (AIME25/26), instruction following (IFBench/IFEval), and competitive-programming code generation (LiveCodeBench). It uses the tokenizer's bundled chat template, Unsloth inference, one visible GPU, resumable JSONL outputs, and line-oriented progress suitable for Papermill logs.

LiveCodeBench has only a test split and must never be used to SFT the code teacher. Its official checker executes generated Python locally; use `LCB_EVAL_ONLY=true` to evaluate saved generations without loading the LLM.

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
print("HF mirror:", os.environ["HF_ENDPOINT"], flush=True)
print("Physical CUDA device exposed to this kernel:", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)

In [ ]:
# Papermill parameters
MODEL_NAME = "Qwen/Qwen3-4B-Base"
OUTPUT_DIR = "results/student_qwen3_4b_base_eval"
NLTK_DATA_DIR = "~/nltk_data"
SEED = 42

LIMIT_AIME25 = None
LIMIT_AIME26 = None
LIMIT_IFBENCH = None
LIMIT_IFEVAL = None
LIMIT_LCB = None

MAX_SEQ_LENGTH = 32768
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0

RUN_MATH = True
RUN_IF = True
RUN_LCB = True
RUN_LCB_EVALUATION = True
LCB_EVAL_ONLY = False  # Papermill: -p LCB_EVAL_ONLY true reuses saved generations without loading the LLM
LCB_RELEASE_VERSION = "release_v6"  # fixed snapshot: 1055 test problems; no train split
LCB_DATASET_REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"  # pinned HF Parquet snapshot
LCB_REPO_DIR = "LiveCodeBench"
LCB_REPO_REVISION = "28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24"
LCB_NUM_PROCESSES = 4
LCB_TIMEOUT_SECONDS = 6
RESUME = True
MAX_CONSECUTIVE_ERRORS = 3
INSTALL_DEPS = False

In [ ]:
# Optional dependency bootstrap. For a CUDA server, preinstall a PyTorch/Unsloth build
# matching the server's CUDA driver whenever possible. This cell never upgrades torch.
import importlib.util
import subprocess
import sys

PACKAGE_BY_MODULE = {
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "datasets": "datasets",
    "nltk": "nltk",
    "tqdm": "tqdm",
    "papermill": "papermill",
}
if not LCB_EVAL_ONLY:
    PACKAGE_BY_MODULE.update({
        "unsloth": "unsloth",
        "ifbench": "git+https://github.com/allenai/IFBench.git@1c40f0c10d9b5c5c2f10a175a28007ebb64f7f4d",
    })
missing = [package for module, package in PACKAGE_BY_MODULE.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_DEPS:
    print("Installing missing Python packages: " + ", ".join(missing), flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
elif missing:
    raise ImportError(
        "Missing packages: " + ", ".join(missing) +
        ". Install them in the CUDA environment, or run Papermill with -p INSTALL_DEPS true."
    )
print("Dependency check passed.", flush=True)

## Runtime and model loading

Because `CUDA_VISIBLE_DEVICES=0` hides physical GPU 1, PyTorch correctly names physical GPU 0 as logical `cuda:0`. The assertions below prevent accidental multi-GPU execution and reject 4-bit loading.

In [ ]:
import gc
import hashlib
import json
import math
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

OFFICIAL_MODEL_REVISION = "906bfd4b4dc7f14ee4320094d8b41684abff8539"
OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
RAW_DIR = OUTPUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}", flush=True)

if LCB_EVAL_ONLY:
    # This path deliberately avoids importing PyTorch/Unsloth or loading the 4B model.
    RUN_MATH = False
    RUN_IF = False
    RUN_LCB = True
    RUN_LCB_EVALUATION = True
    print("[MODE] LCB_EVAL_ONLY: reuse existing LiveCodeBench generations; skip model loading and generation.", flush=True)
else:
    import torch
    from datasets import disable_progress_bars, load_dataset
    from unsloth import FastLanguageModel

    disable_progress_bars()
    assert os.environ.get("CUDA_VISIBLE_DEVICES") == "0", "GPU mask changed before torch import."
    assert torch.cuda.is_available(), "A CUDA GPU is required."
    assert torch.cuda.device_count() == 1, f"Expected exactly one visible GPU, got {torch.cuda.device_count()}."
    torch.cuda.set_device(0)  # logical cuda:0 == physical GPU 0
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print(f"Visible GPU: logical cuda:0 -> {torch.cuda.get_device_name(0)}", flush=True)

    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"Loading {MODEL_NAME} with Unsloth; dtype={DTYPE}; load_in_4bit=False", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=DTYPE,
        load_in_4bit=False, device_map={"": 0},
    )
    FastLanguageModel.for_inference(model)
    model.eval()
    # We pass only max_new_tokens, so clear inherited max_length to avoid duplicate-length warnings.
    model.generation_config.max_length = None
    assert not bool(getattr(model, "is_loaded_in_4bit", False)), "Unexpected 4-bit model."
    quant_method = getattr(getattr(model, "config", None), "quantization_config", None)
    assert quant_method is None, f"Unexpected quantization config: {quant_method}"
    parameter_devices = {str(parameter.device) for parameter in model.parameters()}
    assert parameter_devices == {"cuda:0"}, f"Model is not fully confined to logical cuda:0: {parameter_devices}"
    floating_dtypes = {str(parameter.dtype) for parameter in model.parameters() if parameter.is_floating_point()}
    print(f"Model floating dtypes: {sorted(floating_dtypes)}", flush=True)
    print(f"CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)

## Official chat-template audit

The Base repository defines a Qwen ChatML Jinja template. With `add_generation_prompt=True` and no `enable_thinking` override, a one-turn prompt must end in `<|im_start|>assistant\n`. Unlike Thinking-2507, this pretrained Base checkpoint does not automatically open a `<think>` block.

In [ ]:
# Unsloth may omit chat_template for pretrained Base checkpoints. Always read
# the official Jinja template from the fixed model revision for reproducibility.
if not LCB_EVAL_ONLY:
    from huggingface_hub import hf_hub_download

    tokenizer_config_path = hf_hub_download(
        repo_id=MODEL_NAME, filename="tokenizer_config.json", revision=OFFICIAL_MODEL_REVISION
    )
    official_tokenizer_config = json.loads(Path(tokenizer_config_path).read_text(encoding="utf-8"))
    tokenizer.chat_template = official_tokenizer_config.get("chat_template")
    template_source = f"official tokenizer_config.json at revision {OFFICIAL_MODEL_REVISION}"
    assert tokenizer.chat_template, "Official Qwen3-4B-Base tokenizer config does not contain a chat template."
    audit_messages = [{"role": "user", "content": "What is 1 + 1?"}]
    audit_rendered = tokenizer.apply_chat_template(
        audit_messages, tokenize=False, add_generation_prompt=True
    )
    expected_suffix = "<|im_start|>assistant\n"
    assert audit_rendered.endswith(expected_suffix), (
        "Unexpected Qwen3-4B-Base template ending. Expected "
        + repr(expected_suffix) + " but got " + repr(audit_rendered[-100:])
    )
    im_end_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    generation_eos_token_ids = [int(tokenizer.eos_token_id)]
    if isinstance(im_end_token_id, int) and im_end_token_id >= 0 and im_end_token_id not in generation_eos_token_ids:
        generation_eos_token_ids.append(im_end_token_id)
    template_audit = {
        "model_name": MODEL_NAME, "tokenizer_class": tokenizer.__class__.__name__,
        "eos_token": tokenizer.eos_token, "eos_token_id": tokenizer.eos_token_id,
        "im_end_token_id": im_end_token_id, "generation_eos_token_ids": generation_eos_token_ids,
        "chat_template_source": template_source, "official_model_revision": OFFICIAL_MODEL_REVISION,
        "rendered_example": audit_rendered, "expected_generation_suffix": expected_suffix,
        "checkpoint_type": "pretrained_base", "uses_enable_thinking_argument": False,
        "default_generation_prompt_opens_think_block": False,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
        "official_model_card": "https://huggingface.co/Qwen/Qwen3-4B-Base",
        "official_tokenizer_config": "https://huggingface.co/Qwen/Qwen3-4B-Base/blob/main/tokenizer_config.json",
    }
    (OUTPUT_DIR / "template_audit.json").write_text(
        json.dumps(template_audit, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"chat_template source: {template_source}", flush=True)
    print("Rendered audit prompt:\n" + audit_rendered, flush=True)
else:
    print("[SKIP] Model and chat-template loading are unnecessary in LCB_EVAL_ONLY mode.", flush=True)

## Shared generation and persistence helpers

Each completed sample is appended immediately to JSONL and flushed to disk. A run fingerprint prevents resume mode from silently mixing different models, prompts, or decoding settings. No `tqdm` progress bar is used.

In [ ]:
PROMPT_VERSION = "student-qwen3-4b-base-eval-v1"
fingerprint_payload = {
    "model": MODEL_NAME,
    "prompt_version": PROMPT_VERSION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "seed": SEED,
    "official_model_revision": OFFICIAL_MODEL_REVISION,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Run fingerprint: {RUN_FINGERPRINT}", flush=True)

def format_seconds(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def print_progress(label, done, total, started_at):
    percent = 100.0 * done / total if total else 100.0
    print(
        f"[PROGRESS] {label}: {done}/{total} ({percent:.2f}%) "
        f"elapsed={format_seconds(time.time() - started_at)}",
        flush=True,
    )

def stable_seed(sample_id):
    digest = hashlib.sha256(str(sample_id).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)

def split_thinking(generated_text):
    text = generated_text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False

def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def load_matching_records(path):
    latest = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated JSONL line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == RUN_FINGERPRINT:
                latest[str(record["sample_id"])] = record
    return latest

def limit_dataset(dataset, limit):
    if limit is None:
        return dataset
    return dataset.select(range(min(int(limit), len(dataset))))

def generate_one(user_prompt, max_new_tokens, sample_id, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    assert rendered.endswith(expected_suffix)
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Prompt has {input_tokens} tokens and exceeds MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}.")
    effective_max_new_tokens = min(int(max_new_tokens), int(available))
    if effective_max_new_tokens < max_new_tokens:
        print(
            f"[WARN] {sample_id}: capped max_new_tokens from {max_new_tokens} "
            f"to {effective_max_new_tokens} to fit the context window.",
            flush=True,
        )
    # Unsloth's patched generate() does not accept the Transformers `generator` kwarg.
    # Reset all global RNGs per sample instead, preserving deterministic resume behavior.
    sample_seed = stable_seed(sample_id)
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=effective_max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            eos_token_id=generation_eos_token_ids,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_thinking(raw_generation)
    return {
        "sample_seed": sample_seed,
        "raw_generation": raw_generation,
        "thinking": thinking,
        "final_answer": final_answer,
        "found_think_close": found_think_close,
        "input_tokens": input_tokens,
        "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": effective_max_new_tokens,
        "generation_seconds": time.time() - started,
    }

In [ ]:
# Dataset loading and schema checks. IFBench/IFEval call their sole evaluation split 'train'.
# LiveCodeBench code_generation_lite has only the official test split.
from datasets import disable_progress_bars, load_dataset
disable_progress_bars()
datasets_to_run = {}
if RUN_MATH:
    datasets_to_run["aime25"] = limit_dataset(load_dataset("math-ai/aime25", split="test"), LIMIT_AIME25)
    datasets_to_run["aime26"] = limit_dataset(load_dataset("math-ai/aime26", split="test"), LIMIT_AIME26)
    for name in ("aime25", "aime26"):
        required = {"problem", "answer"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_IF:
    datasets_to_run["ifbench"] = limit_dataset(load_dataset("allenai/IFBench_test", split="train"), LIMIT_IFBENCH)
    datasets_to_run["ifeval"] = limit_dataset(load_dataset("google/IFEval", split="train"), LIMIT_IFEVAL)
    for name in ("ifbench", "ifeval"):
        required = {"key", "prompt", "instruction_id_list", "kwargs"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_LCB:
    lcb_repo_path = Path(LCB_REPO_DIR).expanduser().resolve()
    lcb_package_path = lcb_repo_path / "lcb_runner"
    if not lcb_package_path.is_dir():
        raise FileNotFoundError(
            f"LiveCodeBench source was not found at {lcb_package_path}. Clone the fixed official "
            "repository revision before running this notebook."
        )
    actual_lcb_revision = subprocess.check_output(
        ["git", "-C", str(lcb_repo_path), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual_lcb_revision != LCB_REPO_REVISION:
        raise RuntimeError(
            f"LiveCodeBench revision mismatch: expected {LCB_REPO_REVISION}, got {actual_lcb_revision}. "
            f"Run: git -C {lcb_repo_path} checkout {LCB_REPO_REVISION}"
        )
    if str(lcb_repo_path) not in sys.path:
        sys.path.insert(0, str(lcb_repo_path))
    from lcb_runner.benchmarks.code_generation import CodeGenerationProblem

    # datasets>=4 rejects the legacy loading script used by the fixed official
    # repository. Read the official release_v6 Parquet shards directly instead.
    # Pinning the dataset revision prevents later Hub updates from changing the test set.
    hf_endpoint = os.environ.get("HF_ENDPOINT", "https://huggingface.co").rstrip("/")
    lcb_parquet_urls = [
        f"{hf_endpoint}/datasets/livecodebench/code_generation_lite/resolve/"
        f"{LCB_DATASET_REVISION}/{LCB_RELEASE_VERSION}/test-{index:05d}-of-00009.parquet"
        for index in range(9)
    ]
    print(
        f"Loading LiveCodeBench {LCB_RELEASE_VERSION} from 9 pinned Parquet shards "
        f"at dataset revision {LCB_DATASET_REVISION}.",
        flush=True,
    )
    raw_lcb_dataset = load_dataset(
        "parquet", data_files={"test": lcb_parquet_urls}, split="test"
    )
    lcb_dataset = [CodeGenerationProblem(**row) for row in raw_lcb_dataset]
    lcb_loader_mode = "pinned_parquet_shards"
    lcb_dataset = sorted(lcb_dataset, key=lambda problem: str(problem.question_id))
    if LIMIT_LCB is not None:
        lcb_dataset = lcb_dataset[:min(int(LIMIT_LCB), len(lcb_dataset))]
    datasets_to_run["livecodebench"] = lcb_dataset

dataset_audit = {}
for name, dataset in datasets_to_run.items():
    if name == "livecodebench":
        dataset_audit[name] = {
            "rows": len(dataset), "dataset": "livecodebench/code_generation_lite",
            "release": LCB_RELEASE_VERSION, "split": "test", "has_train_split": False,
            "repository_revision": actual_lcb_revision,
            "dataset_revision": LCB_DATASET_REVISION, "loader_mode": lcb_loader_mode,
        }
    else:
        dataset_audit[name] = {"rows": len(dataset), "columns": dataset.column_names}
(OUTPUT_DIR / "dataset_audit.json").write_text(
    json.dumps(dataset_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(dataset_audit, ensure_ascii=False, indent=2), flush=True)

## Math: AIME25 and AIME26

The primary metric is exact match after extracting the last `\boxed{...}` (with conservative fallbacks) and normalizing the AIME integer answer.

In [ ]:
def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index]
    return None

def extract_math_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches:
        return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""

def normalize_math_answer(value):
    value = str(value).strip()
    value = value.replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value

def run_math_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(example.get("id", index)) for index, example in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] student-qwen3-4b-base/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] student-qwen3-4b-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-qwen3-4b-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        if sample_id in completed:
            done += 1
            continue
        prompt = (
            str(example["problem"]).strip()
            + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
        )
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "problem": example["problem"],
            "gold_answer": str(example["answer"]),
        }
        try:
            result = generate_one(prompt, MAX_NEW_TOKENS, f"{name}:{sample_id}")
            predicted = extract_math_answer(result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "predicted_answer": predicted,
                "correct": normalize_math_answer(predicted) == normalize_math_answer(example["answer"]),
            }
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "correct": False}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-qwen3-4b-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

math_records = {}
if RUN_MATH:
    math_records["aime25"] = run_math_benchmark("aime25", datasets_to_run["aime25"])
    math_records["aime26"] = run_math_benchmark("aime26", datasets_to_run["aime26"])

## Instruction following: IFBench and IFEval

The code below mirrors AllenAI's official strict/loose verifier logic. Crucially, only text after the model's final `</think>` is checked; hidden reasoning must not contaminate length, keyword, or formatting constraints. Primary reported scores are prompt-level loose accuracy, alongside all four official aggregate variants.

In [ ]:
# IFBench's official verifiers require these NLTK resources. Download them before
# importing ifbench so a network problem fails before any new model generations.
import nltk

NLTK_DATA_DIR = Path(NLTK_DATA_DIR).expanduser().resolve()
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(NLTK_DATA_DIR)
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))
NLTK_RESOURCES = {} if LCB_EVAL_ONLY else {
    "punkt_tab": "tokenizers/punkt_tab",
    "stopwords": "corpora/stopwords",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}
for package_name, resource_path in NLTK_RESOURCES.items():
    try:
        nltk.data.find(resource_path)
        print(f"[NLTK] Found {package_name}", flush=True)
    except LookupError:
        print(f"[NLTK] Downloading {package_name} to {NLTK_DATA_DIR}", flush=True)
        last_error = None
        for attempt in range(1, 4):
            try:
                downloaded = nltk.download(
                    package_name, download_dir=str(NLTK_DATA_DIR), quiet=True, raise_on_error=True
                )
                if not downloaded:
                    raise RuntimeError("nltk.download returned False")
                nltk.data.find(resource_path)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                print(f"[NLTK] {package_name} attempt {attempt}/3 failed: {exc!r}", flush=True)
                if attempt < 3:
                    time.sleep(5 * attempt)
        if last_error is not None:
            raise RuntimeError(
                f"Unable to install required NLTK resource {package_name!r}. "
                f"Run `python -m nltk.downloader -d {NLTK_DATA_DIR} {package_name}` and retry."
            ) from last_error

instructions_registry = None
if not LCB_EVAL_ONLY:
    from ifbench import instructions_registry

def response_variants(response):
    lines = response.split("\n")
    remove_first = "\n".join(lines[1:]).strip()
    remove_last = "\n".join(lines[:-1]).strip()
    remove_both = "\n".join(lines[1:-1]).strip()
    return [
        response,
        response.replace("*", ""),
        remove_first,
        remove_last,
        remove_both,
        remove_first.replace("*", ""),
        remove_last.replace("*", ""),
        remove_both.replace("*", ""),
    ]

def check_instruction(instruction_id, kwargs, prompt, candidates):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)
    clean_kwargs = {key: value for key, value in dict(kwargs).items() if value is not None}
    instruction.build_description(**clean_kwargs)
    args = instruction.get_instruction_args()
    if args and "prompt" in args:
        instruction.build_description(prompt=prompt)
    return any(candidate.strip() and instruction.check_following(candidate) for candidate in candidates)

def score_instruction_response(example, response):
    strict = []
    loose = []
    for instruction_id, kwargs in zip(example["instruction_id_list"], example["kwargs"]):
        strict.append(check_instruction(instruction_id, kwargs, example["prompt"], [response]))
        loose.append(check_instruction(instruction_id, kwargs, example["prompt"], response_variants(response)))
    return strict, loose

def run_if_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(example.get("key", index)) for index, example in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] student-qwen3-4b-base/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] student-qwen3-4b-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-qwen3-4b-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("key", index))
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "prompt": example["prompt"],
            "instruction_id_list": example["instruction_id_list"],
            "kwargs": example["kwargs"],
        }
        try:
            result = generate_one(example["prompt"], MAX_NEW_TOKENS, f"{name}:{sample_id}")
            strict, loose = score_instruction_response(example, result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "strict_instruction_results": strict,
                "loose_instruction_results": loose,
                "strict_prompt_pass": all(strict),
                "loose_prompt_pass": all(loose),
            }
            consecutive_errors = 0
        except Exception as exc:
            consecutive_errors += 1
            failed = [False] * len(example["instruction_id_list"])
            record = {
                **base, "status": "error", "error": repr(exc),
                "strict_instruction_results": failed,
                "loose_instruction_results": failed,
                "strict_prompt_pass": False, "loose_prompt_pass": False,
            }
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-qwen3-4b-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation/scoring errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

if_records = {}
if RUN_IF:
    if_records["ifbench"] = run_if_benchmark("ifbench", datasets_to_run["ifbench"])
    if_records["ifeval"] = run_if_benchmark("ifeval", datasets_to_run["ifeval"])

## LiveCodeBench code generation

Generation uses the official generic code-generation instructions plus the checkpoint's verified Qwen ChatML template. The official local Python checker computes pass@1 over the saved programs. Generation is resumable per problem, and `LCB_EVAL_ONLY=true` reruns only the checker.

In [ ]:
# Import the prompt helpers from the fixed official source tree. This revision opens
# its few-shot assets relative to the repository root during import.
# Import only the pinned code-generation prompt module. Importing it normally first
# executes lcb_runner.prompts.__init__, which eagerly imports optional Anthropic-only
# prompt modules even though this local Qwen evaluation does not use Anthropic.
import importlib.util
_notebook_cwd = Path.cwd()
_prompt_module_path = lcb_repo_path / "lcb_runner" / "prompts" / "code_generation.py"
_prompt_spec = importlib.util.spec_from_file_location(
    "_pinned_lcb_code_generation_prompt", _prompt_module_path
)
if _prompt_spec is None or _prompt_spec.loader is None:
    raise ImportError(f"Unable to load LiveCodeBench prompt module: {_prompt_module_path}")
_prompt_module = importlib.util.module_from_spec(_prompt_spec)
try:
    os.chdir(lcb_repo_path)
    _prompt_spec.loader.exec_module(_prompt_module)
finally:
    os.chdir(_notebook_cwd)
PromptConstants = _prompt_module.PromptConstants
get_generic_question_template_answer = _prompt_module.get_generic_question_template_answer
from lcb_runner.lm_styles import LMStyle
from lcb_runner.utils.extraction_utils import extract_code

LCB_PROMPT_VERSION = "official-generic-chat-v1"
lcb_fingerprint_payload = {
    "run_fingerprint": RUN_FINGERPRINT,
    "repository_revision": LCB_REPO_REVISION,
    "dataset_revision": LCB_DATASET_REVISION,
    "release_version": LCB_RELEASE_VERSION,
    "prompt_version": LCB_PROMPT_VERSION,
}
LCB_RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(lcb_fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"LiveCodeBench run fingerprint: {LCB_RUN_FINGERPRINT}", flush=True)

def load_lcb_matching_records(path):
    records = load_matching_records(path)
    return {
        sample_id: record for sample_id, record in records.items()
        if record.get("lcb_run_fingerprint") == LCB_RUN_FINGERPRINT
    }

def make_lcb_prompt(problem):
    # This is LiveCodeBench's official generic user prompt, including its required
    # fenced-Python output format. The official system instruction is passed separately.
    return get_generic_question_template_answer(problem)

def run_lcb_generation(dataset):
    name = "livecodebench"
    output_path = RAW_DIR / "livecodebench.jsonl"
    prior = load_lcb_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(problem.question_id) for problem in dataset}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] student-qwen3-4b-base/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] student-qwen3-4b-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-qwen3-4b-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for problem in dataset:
        sample_id = str(problem.question_id)
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "question_title": problem.question_title,
            "question_content": problem.question_content,
            "platform": problem.platform.value,
            "contest_id": problem.contest_id,
            "contest_date": problem.contest_date.isoformat(),
            "difficulty": problem.difficulty.value,
            "starter_code": problem.starter_code,
        }
        try:
            result = generate_one(
                make_lcb_prompt(problem), MAX_NEW_TOKENS, f"{name}:{sample_id}",
                system_prompt=PromptConstants.SYSTEM_MESSAGE_GENERIC,
            )
            # The official prompt requires fenced code. LiveCodeBench's own extractor
            # returns the last fenced block and returns an empty program if no block exists.
            extracted_code = extract_code(result["final_answer"], LMStyle.CodeQwenInstruct)
            record = {**base, **result, "status": "ok", "extracted_code": extracted_code}
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "extracted_code": ""}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-qwen3-4b-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(
                f"Aborting {name} after {consecutive_errors} consecutive generation errors; "
                f"last error: {record.get('error')}"
            )
    return load_lcb_matching_records(output_path)

lcb_records = {}
if RUN_LCB:
    if LCB_EVAL_ONLY:
        lcb_records = load_lcb_matching_records(RAW_DIR / "livecodebench.jsonl")
        print(f"[SKIP] Generation disabled; loaded {len(lcb_records)} matching LiveCodeBench records.", flush=True)
    else:
        lcb_records = run_lcb_generation(datasets_to_run["livecodebench"])

In [ ]:
# Run the fixed-revision official checker while emitting one durable log line per
# completed problem instead of a terminal-rewriting progress bar.
lcb_eval_path = OUTPUT_DIR / "livecodebench_eval.json"
lcb_eval_status_path = OUTPUT_DIR / "livecodebench_eval_status.json"
lcb_evaluation = {"status": "not_run", "correct": None, "total": None, "pass@1": None}

class LineProgress:
    def __init__(self, *args, total=None, **kwargs):
        self.total = int(total or 0)
        self.done = 0
        self.started = time.time()
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, traceback):
        return False
    def update(self, amount=1):
        self.done += int(amount)
        print_progress("student-qwen3-4b-base/livecodebench-check", self.done, self.total, self.started)

if RUN_LCB and RUN_LCB_EVALUATION:
    lcb_dataset = datasets_to_run["livecodebench"]
    expected_ids = [str(problem.question_id) for problem in lcb_dataset]
    missing_ids = [
        sample_id for sample_id in expected_ids
        if lcb_records.get(sample_id, {}).get("status") != "ok"
    ]
    if missing_ids:
        raise RuntimeError(
            f"LiveCodeBench evaluation requires all generations: {len(missing_ids)} of "
            f"{len(expected_ids)} are missing or failed. Rerun generation first. "
            f"First missing IDs: {missing_ids[:5]}"
        )
    expected_ids_hash = hashlib.sha256(
        json.dumps(expected_ids, ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    lcb_eval_fingerprint = hashlib.sha256(json.dumps({
        "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
        "expected_ids_hash": expected_ids_hash,
        "timeout": LCB_TIMEOUT_SECONDS,
        "repository_revision": LCB_REPO_REVISION,
        "dataset_revision": LCB_DATASET_REVISION,
    }, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    previous_status = {}
    if RESUME and lcb_eval_status_path.is_file() and lcb_eval_path.is_file():
        try:
            previous_status = json.loads(lcb_eval_status_path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[WARN] Ignoring unreadable LiveCodeBench evaluation status: {exc!r}", flush=True)
    if (previous_status.get("status") == "complete"
            and previous_status.get("eval_fingerprint") == lcb_eval_fingerprint):
        lcb_evaluation = previous_status
        print(
            f"[SKIP] LiveCodeBench checker already completed: "
            f"{lcb_evaluation['correct']}/{lcb_evaluation['total']} "
            f"(pass@1={100 * lcb_evaluation['pass@1']:.2f}%).",
            flush=True,
        )
    else:
        import lcb_runner.evaluation.compute_code_generation_metrics as lcb_metric_module
        from lcb_runner.evaluation.pass_k_utils import extract_instance_results
        lcb_metric_module.tqdm = LineProgress
        samples_list = [problem.get_evaluation_sample() for problem in lcb_dataset]
        generations_list = [[lcb_records[str(problem.question_id)]["extracted_code"]] for problem in lcb_dataset]
        official_metrics, official_results, official_metadata = lcb_metric_module.codegen_metrics(
            samples_list,
            generations_list,
            k_list=[1],
            num_process_evaluate=int(LCB_NUM_PROCESSES),
            timeout=int(LCB_TIMEOUT_SECONDS),
            debug=False,
        )
        instance_grades = extract_instance_results(official_results)
        graded = [bool(grades[0]) for grades in instance_grades]
        correct = sum(graded)
        total = len(graded)
        pass_at_1 = float(official_metrics["pass@1"])
        assert math.isclose(pass_at_1, correct / total, rel_tol=0.0, abs_tol=1e-12)
        per_problem = [
            {
                "question_id": expected_ids[index],
                "passed": graded[index],
                "test_results": official_results[index][0],
                "metadata": official_metadata[index][0],
            }
            for index in range(total)
        ]
        eval_payload = {
            "model": MODEL_NAME,
            "release_version": LCB_RELEASE_VERSION,
            "repository_revision": LCB_REPO_REVISION,
            "dataset_revision": LCB_DATASET_REVISION,
            "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
            "eval_fingerprint": lcb_eval_fingerprint,
            "official_metrics": official_metrics,
            "correct": correct,
            "total": total,
            "pass@1": pass_at_1,
            "per_problem": per_problem,
        }
        lcb_eval_path.write_text(json.dumps(eval_payload, ensure_ascii=False, indent=2), encoding="utf-8")
        lcb_evaluation = {
            "status": "complete",
            "correct": correct,
            "total": total,
            "pass@1": pass_at_1,
            "eval_fingerprint": lcb_eval_fingerprint,
            "result_path": str(lcb_eval_path),
        }
        lcb_eval_status_path.write_text(
            json.dumps(lcb_evaluation, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"[RESULT] LiveCodeBench pass@1: {correct}/{total} = {100 * pass_at_1:.2f}%", flush=True)
elif RUN_LCB:
    print("[SKIP] RUN_LCB_EVALUATION=False; generations were saved but not scored.", flush=True)

## Aggregate metrics and bar chart

In [ ]:
metric_rows = []
if LCB_EVAL_ONLY:
    prior_metrics_path = OUTPUT_DIR / "metrics.json"
    if prior_metrics_path.is_file():
        prior_payload = json.loads(prior_metrics_path.read_text(encoding="utf-8"))
        metric_rows = [
            row for row in prior_payload.get("metrics", [])
            if row.get("benchmark") != "LiveCodeBench"
        ]
        print(f"[RESUME] Preserved {len(metric_rows)} completed non-LiveCodeBench metric rows.", flush=True)
    else:
        print("[WARN] No prior metrics.json found; evaluation-only output will contain LiveCodeBench only.", flush=True)

def add_metric(benchmark, domain, metric, numerator, denominator, status="complete", note=""):
    score = numerator / denominator if numerator is not None and denominator else None
    metric_rows.append({
        "benchmark": benchmark, "domain": domain, "metric": metric,
        "score": score, "score_percent": 100 * score if score is not None else None,
        "numerator": numerator, "denominator": denominator, "status": status, "note": note,
    })

for name, records_by_id in math_records.items():
    selected_ids = [str(row.get("id", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    add_metric(name.upper(), "Math", "exact_match", sum(bool(record.get("correct")) for record in records), len(records))

for name, records_by_id in if_records.items():
    selected_ids = [str(row.get("key", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    strict_lists = [record.get("strict_instruction_results", []) for record in records]
    loose_lists = [record.get("loose_instruction_results", []) for record in records]
    instruction_total = sum(len(values) for values in strict_lists)
    add_metric(name.upper(), "Instruction Following", "prompt_level_strict", sum(bool(values) and all(values) for values in strict_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_strict", sum(sum(values) for values in strict_lists), instruction_total)
    add_metric(name.upper(), "Instruction Following", "prompt_level_loose", sum(bool(values) and all(values) for values in loose_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_loose", sum(sum(values) for values in loose_lists), instruction_total)

if RUN_LCB:
    if lcb_evaluation.get("status") == "complete":
        add_metric("LiveCodeBench", "Code", "pass@1", lcb_evaluation["correct"], lcb_evaluation["total"])
    else:
        metric_rows.append({
            "benchmark": "LiveCodeBench", "domain": "Code", "metric": "pass@1",
            "score": None, "score_percent": None, "numerator": None,
            "denominator": lcb_evaluation.get("total"), "status": "not_evaluated",
            "note": "The official LiveCodeBench checker was not run.",
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_csv = OUTPUT_DIR / "metrics.csv"
metrics_json = OUTPUT_DIR / "metrics.json"
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
metrics_payload = {
    "model": MODEL_NAME, "run_fingerprint": RUN_FINGERPRINT,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": metric_rows, "livecodebench_evaluation": lcb_evaluation,
}
metrics_json.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")
display(metrics_df)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

primary_specs = [
    ("AIME25", "exact_match", "AIME25\nExact match"),
    ("AIME26", "exact_match", "AIME26\nExact match"),
    ("IFBENCH", "prompt_level_loose", "IFBench\nPrompt loose"),
    ("IFEVAL", "prompt_level_loose", "IFEval\nPrompt loose"),
    ("LiveCodeBench", "pass@1", "LiveCodeBench\nPass@1"),
]
labels, values, available = [], [], []
for benchmark, metric, label in primary_specs:
    row = metrics_df[(metrics_df["benchmark"] == benchmark) & (metrics_df["metric"] == metric)]
    if row.empty:
        continue
    value = row.iloc[0]["score_percent"]
    labels.append(label)
    available.append(pd.notna(value))
    values.append(float(value) if pd.notna(value) else 0.0)

colors = ["#4C78A8", "#4C78A8", "#F58518", "#F58518", "#54A24B"][:len(labels)]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="#333333", linewidth=0.7)
for bar, value, is_available in zip(bars, values, available):
    if is_available:
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1.2, f"{value:.2f}%", ha="center", va="bottom", fontsize=10)
    else:
        bar.set_facecolor("#D9D9D9")
        bar.set_hatch("//")
        ax.text(bar.get_x() + bar.get_width() / 2, 2.0, "N/A\n(not evaluated)", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(labels)), labels)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 108)
ax.set_title("Qwen3-4B-Base: Student Evaluation")
ax.grid(axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()
figure_path = OUTPUT_DIR / "student_qwen3_4b_base_scores.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Metrics JSON: {metrics_json}", flush=True)
print(f"Metrics CSV:  {metrics_csv}", flush=True)
print(f"Bar chart:    {figure_path}", flush=True)

In [ ]:
gc.collect()
if not LCB_EVAL_ONLY and torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[DONE] student-qwen3-4b-base evaluation notebook completed.", flush=True)